# Phase 1.1: Dataset Preparation - Fusion des Sources Publiques

## Objectif
Fusionner 3 sources de données publiques (EPA AQS, Beijing Multi-Site, UCI Air Quality) vers un schéma canonique unifié.

## Sources de Données
1. **EPA AQS Hourly**: NO2, SO2, CO, PM2.5 + météo (US, 2023)
2. **Beijing Multi-Site**: PM2.5, PM10, SO2, NO2, CO + météo (Chine, 2013-2017)
3. **UCI Air Quality**: NOx, NO2, CO + T/RH (Italie, 2004-2005)

## Sortie
- `ia/data/raw_merged.csv`: Dataset fusionné brut
- Statistiques de couverture par polluant

## Section 1: Configuration de l'Environnement

In [1]:
# ========================================
# SECTION 1: Import des bibliothèques
# ========================================
# Ces imports sont essentiels pour:
# - pandas: manipulation de données (DataFrames)
# - numpy: opérations numériques
# - matplotlib/seaborn: visualisation des données
# - json: manipulation de fichiers JSON pour métadonnées

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from pathlib import Path
from datetime import datetime
import urllib.request
import zipfile
import json

# Désactiver les avertissements (warnings) pour une sortie plus claire
warnings.filterwarnings('ignore')

# Configuration style pour les graphiques
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Afficher les versions des packages pour traçabilité
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Pandas version: 3.0.2
NumPy version: 2.4.4


In [2]:
# ========================================
# SECTION 2: Configuration des chemins
# ========================================
# Établir les répertoires où on va lire/écrire les fichiers
# Path(".") = répertoire courant du notebook

PROJECT_ROOT = Path("../").resolve()  # Remonter d'un niveau (ia/)
DATA_DIR = PROJECT_ROOT / "data"      # Répertoire pour les fichiers de données
MODELS_DIR = PROJECT_ROOT / "models"  # Répertoire pour les modèles et résultats

# Créer les répertoires s'ils n'existent pas (ne fait rien si déjà existants)
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print(f"📁 PROJECT_ROOT: {PROJECT_ROOT}")
print(f"📁 DATA_DIR: {DATA_DIR}")
print(f"📁 MODELS_DIR: {MODELS_DIR}")

📁 PROJECT_ROOT: C:\Users\melik\Desktop\pollution_monitoring\ia
📁 DATA_DIR: C:\Users\melik\Desktop\pollution_monitoring\ia\data
📁 MODELS_DIR: C:\Users\melik\Desktop\pollution_monitoring\ia\models


In [3]:
# ========================================
# SECTION 3: Définition du schéma canonique
# ========================================
# Le "schéma canonique" est le format standard auquel on convertit TOUTES
# les sources de données. Cela permet de combiner EPA, Beijing, UCI facilement.

CANONICAL_SCHEMA = {
    # Identifiants temporels et spatiaux
    'timestamp_utc': 'datetime64[ns]',  # Date/heure en UTC pour traçabilité
    'site_id': 'object',                 # ID unique du site de monitoring
    'site_name': 'object',               # Nom du site (ex: "Beijing_Aotizhongxin")
    
    # Données de polluants et unités
    'pollutant': 'object',               # NO2, SO2, CO, PM25, PM10, CO2, COV, etc.
    'value': 'float64',                  # Valeur mesurée du polluant
    'unit': 'object',                    # Unité (ppb, ug/m3, ppm, etc.)
    
    # Données météorologiques (contexte)
    'temperature_c': 'float64',          # Température en °C
    'humidity_percent': 'float64',       # Humidité relative en %
    'pressure_hpa': 'float64',           # Pression barométrique en hPa
    'wind_speed_ms': 'float64',          # Vitesse du vent en m/s
    
    # Traçabilité des sources
    'source_name': 'object',             # EPA_AQS, Beijing_MultiSite, UCI_AirQuality
    'source_id': 'int64'                 # ID numérique de la source (1, 2, 3)
}

# Limites physiques de référence pour le contrôle qualité
# Si une valeur sort de ces limites, elle est probablement aberrante (erreur capteur)
POLLUTANT_BOUNDS = {
    'CO2': {'min': 300, 'max': 5000, 'unit': 'ppm'},
    'NOx': {'min': 0, 'max': 500, 'unit': 'ppb'},
    'NO2': {'min': 0, 'max': 400, 'unit': 'ppb'},
    'SO2': {'min': 0, 'max': 300, 'unit': 'ppb'},
    'CO': {'min': 0, 'max': 10000, 'unit': 'ppb'},
    'PM25': {'min': 0, 'max': 500, 'unit': 'ug/m3'},
    'PM10': {'min': 0, 'max': 600, 'unit': 'ug/m3'},
    'COV': {'min': 0, 'max': 1000, 'unit': 'ppb'}
}

print("✅ Schéma canonique défini")
print(f"   Polluants monitorés: {', '.join(POLLUTANT_BOUNDS.keys())}")

✅ Schéma canonique défini
   Polluants monitorés: CO2, NOx, NO2, SO2, CO, PM25, PM10, COV


## Section 2: Chargement des Données Brutes

In [ ]:
# ========================================
# SOURCE A: EPA AQS Hourly Data (USA) - Téléchargement réel
# ========================================
# Ce bloc télécharge les fichiers listés par EPA (file_list.csv),
# sauvegarde les archives dans ia/data/epa_raw/ puis extrait et concatène les CSV pertinents.

print("🔄 Ingestion EPA AQS (download + map) ...\n")

epa_raw_dir = DATA_DIR / 'epa_raw'
epa_raw_dir.mkdir(parents=True, exist_ok=True)

file_list_url = 'https://aqs.epa.gov/aqsweb/airdata/file_list.csv'

def download_file(url, dest_path):
    if dest_path.exists():
        print(f'↺ Déjà présent: {dest_path.name}')
        return
    print(f'→ Téléchargement: {url}')
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    with open(dest_path, 'wb') as f:
        f.write(r.content)

# Lister les fichiers disponibles chez EPA
try:
    fl = pd.read_csv(file_list_url)
    print(f'file_list téléchargé: {len(fl)} entrées')
except Exception as e:
    print('Impossible de récupérer file_list.csv depuis EPA:', e)
    fl = pd.DataFrame()

# Filtrer candidates horaires pour les polluants d'intérêt (NO2, SO2, CO, PM2.5)
candidates = []
if not fl.empty:
    names = fl['file_name'].astype(str)
    mask = names.str.contains('hourly', case=False, na=False) | names.str.contains('daily', case=False, na=False)
    candidates = names[mask].tolist()
    print(f'Candidates EPA (hourly/daily): {len(candidates)}')

# Télécharger et extraire un petit nombre de fichiers récents (par défaut 3 pour éviter surcharge)
epa_dfs = []
for fn in candidates[:6]:
    try:
        url = 'https://aqs.epa.gov/aqsweb/airdata/' + fn
        dest = epa_raw_dir / fn
        download_file(url, dest)
        # Extraire CSV(s) du zip et charger
        if dest.suffix.lower() == '.zip':
            with zipfile.ZipFile(dest, 'r') as z:
                for name in z.namelist():
                    if name.lower().endswith('.csv'):
                        with z.open(name) as f:
                            try:
                                df = pd.read_csv(f, low_memory=False)
                                epa_dfs.append(df)
                            except Exception as e:
                                print('Erreur lecture CSV dans ZIP:', e)
        else:
            try:
                df = pd.read_csv(dest, low_memory=False)
                epa_dfs.append(df)
            except Exception as e:
                print('Erreur lecture fichier EPA:', e)
    except Exception as e:
        print('Échec téléchargement/extraction:', fn, e)

# Concaténer si des frames ont été chargés
if epa_dfs:
    epa_raw = pd.concat(epa_dfs, ignore_index=True, sort=False)
    print(f'✅ EPA raw concaténé: {len(epa_raw)} lignes')
else:
    epa_raw = pd.DataFrame()
    print('⚠️ Aucun fichier EPA chargé automatiquement — vérifier connexion ou file_list.csv')

# Mapper vers schéma canonique si colonnes attendues présentes
def map_epa_to_canonical(df):
    df2 = df.copy()
    # Colonnes usuelles: sample_measurement, date_gmt/date_local, time_local, parameter_name, units_of_measure
    if 'sample_measurement' in df2.columns:
        df2 = df2.rename(columns={'sample_measurement':'value'})
    if 'date_gmt' in df2.columns and 'time_gmt' in df2.columns:
        df2['timestamp_utc'] = pd.to_datetime(df2['date_gmt'].astype(str) + ' ' + df2['time_gmt'].astype(str), errors='coerce')
    elif 'date_local' in df2.columns and 'time_local' in df2.columns:
        df2['timestamp_utc'] = pd.to_datetime(df2['date_local'].astype(str) + ' ' + df2['time_local'].astype(str), errors='coerce')
    elif 'date_local' in df2.columns:
        df2['timestamp_utc'] = pd.to_datetime(df2['date_local'], errors='coerce')
    if 'parameter_name' in df2.columns:
        df2['pollutant'] = df2['parameter_name']
    if 'units_of_measure' in df2.columns:
        df2['unit'] = df2['units_of_measure']
    # site id heuristique
    for c in ['site_id','site_number','monitor_id']:
        if c in df2.columns and 'site_id' not in df2.columns:
            df2['site_id'] = df2[c]
    keep = [c for c in ['timestamp_utc','site_id','pollutant','value','unit'] if c in df2.columns]
    return df2[keep]

epa_df = map_epa_to_canonical(epa_raw) if not epa_raw.empty else pd.DataFrame()
if not epa_df.empty:
    epa_df['source_name'] = 'EPA_AQS'
    epa_df['source_id'] = 1
    print(f'✅ EPA mappé: {len(epa_df)} lignes — polluants: {epa_df[
].unique()}')
else:
    print('⚠️ EPA non chargé / mappé')

🔄 Chargement EPA AQS...

✅ EPA AQS chargé: 2000 lignes
   Polluants: <StringArray>
['CO', 'SO2', 'NO2', 'PM25']
Length: 4, dtype: str
   Période: 2023-01-01 00:00:00 à 2023-03-25 07:00:00


In [ ]:
# ========================================
# SOURCE B: Beijing Multi-Site Air Quality Data - Téléchargement (template)
# ========================================
# Le jeu de données Beijing Multi-Site peut être volumineux et disponible via plusieurs mirrors.
# Ici on propose une fonction générique pour télécharger une archive fournie par URL.

print("🔄 Ingestion Beijing Multi-Site (download template) ...\n")

beijing_raw_dir = DATA_DIR / 'beijing_raw'
beijing_raw_dir.mkdir(parents=True, exist_ok=True)

def download_from_url(url, dest_dir, overwrite=False):
    dest = dest_dir / Path(url).name
    if dest.exists() and not overwrite:
        print(f'↺ Déjà présent: {dest.name}')
        return dest
    print(f'→ Téléchargement: {url}')
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    with open(dest, 'wb') as f:
        f.write(r.content)
    return dest

# Exemple: si vous avez une URL fiable pour Beijing, renseignez-la ci-dessous.
beijing_url = None  # ex: 'https://.../BeijingMultiSite.zip'

beijing_df = pd.DataFrame()
if beijing_url:
    try:
        zpath = download_from_url(beijing_url, beijing_raw_dir)
        if zpath.suffix.lower() == '.zip':
            with zipfile.ZipFile(zpath, 'r') as z:
                for name in z.namelist():
                    if name.lower().endswith('.csv'):
                        with z.open(name) as f:
                            try:
                                tmp = pd.read_csv(f, low_memory=False)
                                beijing_df = pd.concat([beijing_df, tmp], ignore_index=True, sort=False)
                            except Exception as e:
                                print('Erreur lecture CSV Beijing:', e)
        else:
            beijing_df = pd.read_csv(zpath, low_memory=False)
        print(f'✅ Beijing chargé depuis URL: {len(beijing_df)} lignes')
    except Exception as e:
        print('Échec téléchargement Beijing:', e)
else:
    print('⚠️ Aucun URL Beijing fourni — veuillez définir `beijing_url` ou déposer localement les fichiers dans ia/data/beijing_raw/')

# Mapping: adapter selon la structure du fichier téléchargé
if not beijing_df.empty:
    # Tentative de renommage vers schéma canonique (ajuster si nécessaire)
    if 'timestamp' in beijing_df.columns:
        beijing_df['timestamp_utc'] = pd.to_datetime(beijing_df['timestamp'], errors='coerce')
    print('✅ Beijing mappé (si colonnes attendues présentes)')

🔄 Chargement Beijing Multi-Site...

✅ Beijing Multi-Site chargé: 2000 lignes
   Sites: <StringArray>
['Aotizhongxin', 'Chaoyang', 'Changping', 'Daxing']
Length: 4, dtype: str
   Polluants: PM25, PM10, SO2, NO2, CO


In [ ]:
# ========================================
# SOURCE C: UCI Air Quality Dataset (Padova) - Téléchargement réel
# ========================================
# URL officiel UCI (AirQualityUCI): https://archive.ics.uci.edu/ml/machine-learning-databases/00360/

print("🔄 Ingestion UCI AirQualityUCI (Padova) ...\n")

uci_raw_dir = DATA_DIR / 'uci_raw'
uci_raw_dir.mkdir(parents=True, exist_ok=True)

uci_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip'

def download_and_extract_zip(url, dest_dir):
    dest = dest_dir / Path(url).name
    if not dest.exists():
        print(f'→ Téléchargement: {url}')
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(dest, 'wb') as f:
            f.write(r.content)
    else:
        print(f'↺ Déjà présent: {dest.name}')
    # Extraire et tenter de repérer un CSV
    dfs = []
    try:
        with zipfile.ZipFile(dest, 'r') as z:
            for name in z.namelist():
                if name.lower().endswith('.csv') or name.lower().endswith('.txt'):
                    with z.open(name) as f:
                        try:
                            tmp = pd.read_csv(f, sep=';', decimal=',', low_memory=False)
                            dfs.append(tmp)
                        except Exception as e:
                            try:
                                tmp = pd.read_csv(f, low_memory=False)
                                dfs.append(tmp)
                            except Exception as e2:
                                print('Erreur lecture UCI file inside zip:', e2)
    except Exception as e:
        print('Erreur extraction UCI zip:', e)
    return pd.concat(dfs, ignore_index=True, sort=False) if dfs else pd.DataFrame()

uci_df = download_and_extract_zip(uci_url, uci_raw_dir)
if not uci_df.empty:
    print(f'✅ UCI raw chargé: {len(uci_df)} lignes — colonnes: {list(uci_df.columns)[:10]}')
    # Tentative de mappage vers schéma canonique (adapter selon structure du CSV)
    # Exemple basique: certaines colonnes du dataset UCI utilisent séparateurs et décimales locales
    # L'utilisateur devra valider les colonnes effectivement présentes et ajuster le mapping si nécessaire.
else:
    print('⚠️ UCI AirQualityUCI non téléchargé automatiquement — vérifier URL ou connexion')

## Section 3: Transformation vers Schéma Canonique

In [ ]:
# Transformation EPA → schéma canonique

print("🔄 Transformation EPA AQS...")

epa_canonical = pd.DataFrame()
epa_canonical['timestamp_utc'] = epa_df['date_observed']
epa_canonical['site_id'] = epa_df['site_id']
epa_canonical['site_name'] = epa_df['county_name']
epa_canonical['pollutant'] = epa_df['pollutant']
epa_canonical['value'] = epa_df['observation_value']
epa_canonical['unit'] = 'ppb'  # EPA AQS en ppb pour NO2/SO2/CO
epa_canonical['temperature_c'] = epa_df['temperature']
epa_canonical['humidity_percent'] = epa_df['relative_humidity']
epa_canonical['pressure_hpa'] = epa_df['barometric_pressure']
epa_canonical['wind_speed_ms'] = epa_df['wind_speed']
epa_canonical['source_name'] = epa_df['source_name']
epa_canonical['source_id'] = epa_df['source_id']

print(f"✅ EPA transformé: {len(epa_canonical)} lignes")
print(f"\nAperçu:")
print(epa_canonical.head())

In [ ]:
# Transformation Beijing → schéma canonique (format long)

print("🔄 Transformation Beijing Multi-Site...")

beijing_long = []
pollutants_beijing = ['PM25', 'PM10', 'SO2', 'NO2', 'CO']

for pollutant in pollutants_beijing:
    subset = beijing_df[['timestamp', 'site_id', pollutant, 'TEMP', 'RH', 'PRES', 'Ws', 'source_name', 'source_id']].copy()
    subset.rename(columns={pollutant: 'value'}, inplace=True)
    subset['pollutant'] = pollutant.replace('PM25', 'PM25').replace('PM10', 'PM10').replace('NO2', 'NO2').replace('SO2', 'SO2').replace('CO', 'CO')
    subset['unit'] = 'ug/m3' if 'PM' in pollutant else 'ppb'
    beijing_long.append(subset)

beijing_canonical = pd.concat(beijing_long, ignore_index=True)
beijing_canonical.rename(columns={
    'timestamp': 'timestamp_utc',
    'TEMP': 'temperature_c',
    'RH': 'humidity_percent',
    'PRES': 'pressure_hpa',
    'Ws': 'wind_speed_ms'
}, inplace=True)
beijing_canonical['site_name'] = beijing_canonical['site_id']

print(f"✅ Beijing transformé: {len(beijing_canonical)} lignes")
print(f"\nAperçu:")
print(beijing_canonical.head())

In [ ]:
# Transformation UCI → schéma canonique

print("🔄 Transformation UCI Air Quality...")

uci_long = []
pollutants_uci = ['NOx', 'NO2', 'CO']

for pollutant in pollutants_uci:
    subset = uci_df[['date', 'site_id', pollutant, 'T', 'RH']].copy()
    subset.rename(columns={pollutant: 'value'}, inplace=True)
    subset['pollutant'] = pollutant
    subset['unit'] = 'ppb'
    uci_long.append(subset)

uci_canonical = pd.concat(uci_long, ignore_index=True)
uci_canonical.rename(columns={
    'date': 'timestamp_utc',
    'T': 'temperature_c',
    'RH': 'humidity_percent'
}, inplace=True)
uci_canonical['pressure_hpa'] = np.nan  # Non disponible
uci_canonical['wind_speed_ms'] = np.nan  # Non disponible
uci_canonical['site_name'] = uci_canonical['site_id']
uci_canonical['source_name'] = 'UCI_AirQuality'
uci_canonical['source_id'] = 3

print(f"✅ UCI transformé: {len(uci_canonical)} lignes")
print(f"\nAperçu:")
print(uci_canonical.head())

## Section 4: Fusion et Normalisation

In [ ]:
# Fusionner tous les datasets

print("🔄 Fusion des 3 sources...\n")

# S'assurer que toutes les colonnes du schéma canonique sont présentes
for df in [epa_canonical, beijing_canonical, uci_canonical]:
    for col in CANONICAL_SCHEMA.keys():
        if col not in df.columns:
            df[col] = np.nan

# Concaténer
merged_df = pd.concat([
    epa_canonical[list(CANONICAL_SCHEMA.keys())],
    beijing_canonical[list(CANONICAL_SCHEMA.keys())],
    uci_canonical[list(CANONICAL_SCHEMA.keys())]
], ignore_index=True)

# Appliquer les types
for col, dtype in CANONICAL_SCHEMA.items():
    try:
        merged_df[col] = merged_df[col].astype(dtype)
    except:
        pass  # Laisser certaines conversions si échec

print(f"✅ Fusion complète: {len(merged_df)} lignes total")
print(f"   Période: {merged_df['timestamp_utc'].min()} à {merged_df['timestamp_utc'].max()}")
print(f"\n📊 Distribution par source:")
print(merged_df['source_name'].value_counts())
print(f"\n📊 Distribution par polluant:")
print(merged_df['pollutant'].value_counts())

## Section 5: Contrôle Qualité

In [ ]:
# Contrôle 1: Doublons

print("🔍 Contrôle 1: DOUBLONS\n")

duplicates = merged_df.duplicated(
    subset=['timestamp_utc', 'site_id', 'pollutant'],
    keep=False
)

n_duplicates = duplicates.sum()
print(f"Doublons trouvés: {n_duplicates}")

if n_duplicates > 0:
    print(f"\n⚠️  Exemples de doublons:")
    print(merged_df[duplicates].head(10))
    # Supprimer
    merged_df = merged_df.drop_duplicates(
        subset=['timestamp_utc', 'site_id', 'pollutant'],
        keep='first'
    )
    print(f"\n✅ Doublons supprimés. Lignes restantes: {len(merged_df)}")
else:
    print("✅ Aucun doublon détecté")

In [ ]:
# Contrôle 2: Valeurs physiques aberrantes

print("🔍 Contrôle 2: VALEURS ABERRANTES\n")

out_of_bounds = []

for pollutant, bounds in POLLUTANT_BOUNDS.items():
    mask = merged_df['pollutant'] == pollutant
    subset = merged_df[mask]
    
    if len(subset) > 0:
        out_min = subset['value'] < bounds['min']
        out_max = subset['value'] > bounds['max']
        
        n_out = out_min.sum() + out_max.sum()
        
        if n_out > 0:
            print(f"⚠️  {pollutant}: {n_out} valeurs hors plage [{bounds['min']}, {bounds['max']}] {bounds['unit']}")
            out_of_bounds.append(subset[out_min | out_max].index)
        else:
            print(f"✅ {pollutant}: OK ({len(subset)} points)")

# Marquer les valeurs aberrantes (ne pas supprimer)
merged_df['is_outlier'] = False
if out_of_bounds:
    all_outliers = np.concatenate(out_of_bounds)
    merged_df.loc[all_outliers, 'is_outlier'] = True
    print(f"\n⚠️  Total valeurs aberrantes marquées: {merged_df['is_outlier'].sum()}")

In [ ]:
# Contrôle 3: Taux de manquants

print("🔍 Contrôle 3: TAUX DE MANQUANTS\n")

missing_pct = merged_df.isnull().sum() / len(merged_df) * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

if len(missing_pct) > 0:
    print("Colonnes avec valeurs manquantes:")
    for col, pct in missing_pct.items():
        print(f"  {col:20s}: {pct:6.2f}%")
else:
    print("✅ Aucune valeur manquante")

# Ajouter flag
merged_df['imputed'] = False

In [ ]:
# Visualisation 1: Couverture par polluant et source

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme: Nombre de points par polluant
pollutant_counts = merged_df['pollutant'].value_counts()
axes[0].bar(pollutant_counts.index, pollutant_counts.values, color='steelblue')
axes[0].set_title('Distribution des Données par Polluant', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Polluant')
axes[0].set_ylabel('Nombre de points')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart: Distribution par source
source_counts = merged_df['source_name'].value_counts()
axes[1].pie(source_counts.values, labels=source_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Distribution des Données par Source', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(DATA_DIR / 'coverage_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Graphiques sauvegardés")

## Section 6: Statistiques Résumées et Export

In [ ]:
# Statistiques par polluant

print("📊 STATISTIQUES PAR POLLUANT\n")

for pollutant in merged_df['pollutant'].unique():
    if pd.notna(pollutant):
        subset = merged_df[merged_df['pollutant'] == pollutant]['value']
        if len(subset) > 0:
            print(f"{pollutant}:")
            print(f"  Comptage: {len(subset)}")
            print(f"  Mean:  {subset.mean():.2f}")
            print(f"  Std:   {subset.std():.2f}")
            print(f"  Min:   {subset.min():.2f}")
            print(f"  Max:   {subset.max():.2f}")
            print()

In [ ]:
# Exporter le dataset fusionné

output_path = DATA_DIR / 'raw_merged.csv'
merged_df.to_csv(output_path, index=False)

print(f"✅ Dataset fusionné exporté à: {output_path}")
print(f"   Format: CSV")
print(f"   Taille: {len(merged_df):,} lignes × {len(merged_df.columns)} colonnes")
print(f"   Poids: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
# Exporter métadonnées et rapport

metadata = {
    'created_at': datetime.now().isoformat(),
    'total_rows': len(merged_df),
    'sources': merged_df['source_name'].unique().tolist(),
    'pollutants': merged_df['pollutant'].unique().tolist(),
    'date_range': {
        'min': merged_df['timestamp_utc'].min().isoformat() if pd.notna(merged_df['timestamp_utc'].min()) else None,
        'max': merged_df['timestamp_utc'].max().isoformat() if pd.notna(merged_df['timestamp_utc'].max()) else None
    },
    'quality_flags': {
        'duplicates_removed': n_duplicates,
        'outliers_marked': merged_df['is_outlier'].sum(),
        'missing_values': missing_pct.to_dict() if len(missing_pct) > 0 else {}
    },
    'source_distribution': merged_df['source_name'].value_counts().to_dict()
}

metadata_path = DATA_DIR / 'raw_merged_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"✅ Métadonnées exportées à: {metadata_path}")
print(f"\nRésumé du dataset fusionné:")
print(json.dumps(metadata, indent=2, default=str))

## ✅ Prochaines étapes

1. **Notebook 02**: Nettoyage et preprocessing (`02_data_cleaning_preprocessing.ipynb`)
   - Interpolation des trous
   - Lissage EMA
   - Feature engineering
   - Output: `ia/data/cleaned_features.csv`

2. **Notebook 03**: Génération données synthétiques (`03_synthetic_data_generation.ipynb`)
   - CO2, PM10, COV synthétiques
   - Output: `ia/data/training_dataset.csv`

3. **Notebooks 04-07**: Entraînement modèles
   - Isolation Forest
   - LSTM (horizons 1h et 24h)